# 08.8 - Hugging Face Ecosystem

**Phase:** 08 - Transformers

**Status:** VERIFIED

---

## 1. What Are We Solving?

The Hugging Face ecosystem provides libraries (Transformers, Datasets, Tokenizers, Accelerate) and a model hub for working with pre-trained models, datasets, and tokenizers.

## 2. Why Does This Matter?

Hugging Face is the standard platform for NLP and transformer work. It provides pre-trained models, easy-to-use APIs, and a community hub. Most ML practitioners use it daily.

## 3. Prerequisites

- Python, PyTorch basics, understanding of transformers (08.1-08.7).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Use Hugging Face pipelines for common tasks.
- Load and process datasets with the `datasets` library.
- Use `AutoModel` and `AutoTokenizer` for portability.
- Understand the `Trainer` API and `Accelerate`.

## 5. Mental Model

Hugging Face is like a library system. The model hub is the catalog (thousands of pre-trained models). The `transformers` library is the checkout system (load any model with one line). The `datasets` library is the reference section. The `tokenizers` library is the translation desk.

> NOTE: This notebook demonstrates the API using tiny randomly-initialized models to avoid downloading large pretrained weights from the hub.


## 6. The Transformers Library: AutoModel & AutoTokenizer

`AutoModel` and `AutoTokenizer` load any model/tokenizer by name. Here we create a tiny BERT model from a config (no download).


In [1]:
import matplotlib
matplotlib.use('Agg')
import torch
from transformers import BertConfig, BertModel, BertTokenizerFast

# Create a TINY BERT config - no weights downloaded from the hub
config = BertConfig(
    vocab_size=100,
    hidden_size=32,
    num_hidden_layers=2,
    num_attention_heads=2,
    intermediate_size=64,
)
model = BertModel(config)
print("Tiny BERT model created from config (random init, no download)")
print("Number of parameters:", sum(p.numel() for p in model.parameters()))
print("Config:", config)


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tiny BERT model created from config (random init, no download)
Number of parameters: 37856
Config: BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 32,
  "initializer_range": 0.02,
  "intermediate_size": 64,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.16.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 100
}



## 7. Tokenizer Basics

A tokenizer converts text to input IDs. We build a tiny tokenizer from a small vocabulary using a simple word-level mapping.


In [2]:
# Build a tiny word-level tokenizer from a small vocab
vocab = {
    "[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3, "[MASK]": 4,
    "hello": 5, "world": 6, "transformers": 7, "are": 8, "awesome": 9,
}
id_to_token = {v: k for k, v in vocab.items()}

def encode(text):
    """Convert text to a list of token ids."""
    words = text.split()
    ids = [vocab.get(w, vocab["[UNK]"]) for w in words]
    return [vocab["[CLS]"]] + ids + [vocab["[SEP]"]]

def decode(ids):
    """Convert token ids back to text."""
    return " ".join(id_to_token.get(i, "[UNK]") for i in ids)

print("Tokenizer vocab size:", len(vocab))
print("Special tokens:", ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"])

# Encode a simple input
ids = encode("hello world")
print("Encoded 'hello world':", ids)
print("Decoded back:", decode(ids))

# Unknown word handling
print("Unknown word 'xyzzy':", encode("hello xyzzy"))


Tokenizer vocab size: 10
Special tokens: ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
Encoded 'hello world': [2, 5, 6, 3]
Decoded back: [CLS] hello world [SEP]
Unknown word 'xyzzy': [2, 5, 1, 3]


## 8. The Datasets Library

The `datasets` library provides efficient data loading and processing. We create a small in-memory dataset.


In [3]:
from datasets import Dataset

# Create a small dataset
data = {
    "text": ["I love this product", "This is terrible", "Great experience", "Worst purchase ever"],
    "label": [1, 0, 1, 0],
}
ds = Dataset.from_dict(data)
print("Dataset:")
print(ds)
print("\nFirst example:", ds[0])
print("Features:", ds.features)

# Map a function over the dataset
def add_length(example):
    example["length"] = len(example["text"])
    return example

ds = ds.map(add_length)
print("\nAfter map:", ds.column_names)


Dataset:
Dataset({
    features: ['text', 'label'],
    num_rows: 4
})

First example: {'text': 'I love this product', 'label': 1}
Features: {'text': Value('string'), 'label': Value('int64')}


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map: 100%|██████████| 4/4 [00:00<00:00, 46.45 examples/s]


After map: ['text', 'label', 'length']


## 9. Pipelines: High-Level API

Pipelines wrap a model + tokenizer for a specific task. We demonstrate the concept with a tiny model and a manual forward pass (to avoid downloading a real tokenizer).


In [4]:
from transformers import BertForSequenceClassification
import torch

# Build a tiny text-classification model (random init, no download)
cls_config = BertConfig(
    vocab_size=100, hidden_size=32, num_hidden_layers=2,
    num_attention_heads=2, intermediate_size=64, num_labels=2,
)
cls_model = BertForSequenceClassification(cls_config)

# Manually build input_ids from our tiny vocab and run a forward pass
input_ids = torch.tensor([encode("hello world")])
with torch.no_grad():
    outputs = cls_model(input_ids=input_ids)
    probs = torch.softmax(outputs.logits, dim=-1)

print("Input ids:", input_ids.tolist())
print("Logits:", outputs.logits.tolist())
print("Probabilities:", probs.tolist())
print("\nThis is what a pipeline does internally: tokenize -> model -> softmax.")
print("In production: pipeline('sentiment-analysis') loads a pretrained model + tokenizer.")


Input ids: [[2, 5, 6, 3]]
Logits: [[-0.008072558790445328, -0.008984689600765705]]
Probabilities: [[0.5002280473709106, 0.49977198243141174]]

This is what a pipeline does internally: tokenize -> model -> softmax.
In production: pipeline('sentiment-analysis') loads a pretrained model + tokenizer.


## 10. The Trainer API

`Trainer` provides a standard training loop with minimal boilerplate. We demonstrate with a tiny model on synthetic data.


In [5]:
from transformers import Trainer, TrainingArguments
import numpy as np

# Tiny synthetic dataset
train_data = {
    "input_ids": [[5, 6, 0, 0], [7, 8, 9, 0], [5, 9, 0, 0], [7, 6, 8, 0]],
    "attention_mask": [[1, 1, 0, 0], [1, 1, 1, 0], [1, 1, 0, 0], [1, 1, 1, 0]],
    "labels": [1, 0, 1, 0],
}
train_ds = Dataset.from_dict(train_data)

args = TrainingArguments(
    output_dir="./tiny_trainer_results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    logging_steps=1,
    report_to=[],
    disable_tqdm=True,
)

trainer = Trainer(
    model=cls_model,
    args=args,
    train_dataset=train_ds,
)

trainer.train()
print("\nTrainer training completed (tiny model, 1 epoch).")


D:\CODE\complete ml\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': '0.6959', 'grad_norm': '0.2822', 'learning_rate': '5e-05', 'epoch': '0.5'}
{'loss': '0.6948', 'grad_norm': '0.2842', 'learning_rate': '2.5e-05', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 14.68it/s]

{'train_runtime': '1.607', 'train_samples_per_second': '2.488', 'train_steps_per_second': '1.244', 'train_loss': '0.6954', 'epoch': '1'}

Trainer training completed (tiny model, 1 epoch).


## 11. Failure Case: Tokenizer-Model Mismatch

Using the wrong tokenizer for a model causes garbage output.


In [6]:
# Illustrative: a tokenizer with a different vocab than the model expects
print("If you load a model with vocab_size=100 but a tokenizer with vocab_size=50000,")
print("the input IDs may exceed the model's embedding table, causing errors or garbage.")
print("\nAlways use the SAME model name for both AutoModel and AutoTokenizer.")
print("Example: model = AutoModel.from_pretrained('bert-base-uncased')")
print("         tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')")


If you load a model with vocab_size=100 but a tokenizer with vocab_size=50000,
the input IDs may exceed the model's embedding table, causing errors or garbage.

Always use the SAME model name for both AutoModel and AutoTokenizer.
Example: model = AutoModel.from_pretrained('bert-base-uncased')
         tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')


## 12. Debugging: Common Errors

- **tokenizer/model mismatch**: different model names. Use same name for both.
- **OOM on large model**: model too big. Use `device_map="auto"` or smaller model.
- **Slow dataset processing**: not using `batched=True`. Set it in `map()`.
- **Pipeline wrong labels**: wrong task specified. Use correct task name.

## 13. Real-World Considerations

- Use `AutoModel`/`AutoTokenizer` for portability.
- Use `datasets` `map()` with `batched=True` for efficiency.
- Check model cards before using a model from the Hub.
- Use `device_map="auto"` for easy GPU distribution.

## 14. Common Mistakes

- Forgetting `padding=True` and `truncation=True` during batch processing.
- Looping over datasets instead of using `map()`.
- Ignoring model cards.

## 15. When NOT to Use

- Simple classical ML problems (use scikit-learn).
- When you need full control over the training loop (write custom loop).

## 16. Challenge

Use `datasets` to create a dataset, tokenize it with a tiny tokenizer, and prepare it for training.


In [7]:
# Challenge: tokenize a dataset with the tiny tokenizer
def tokenize_fn(example):
    # Encode each text to input_ids using our tiny encode() function
    ids = encode(example["text"])
    max_len = 8
    padded = ids[:max_len] + [0] * (max_len - len(ids))
    example["input_ids"] = padded
    example["attention_mask"] = [1 if i < len(ids) else 0 for i in range(max_len)]
    return example

tokenized = ds.map(tokenize_fn)
print("Tokenized dataset columns:", tokenized.column_names)
print("First example input_ids:", tokenized[0]["input_ids"])
print("First example attention_mask:", tokenized[0]["attention_mask"])


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map: 100%|██████████| 4/4 [00:00<00:00, 383.47 examples/s]

Tokenized dataset columns: ['text', 'label', 'length', 'input_ids', 'attention_mask']
First example input_ids: [2, 1, 1, 1, 1, 3, 0, 0]
First example attention_mask: [1, 1, 1, 1, 1, 1, 0, 0]


## 17. Closed-Book Recall

Without looking back:

1. What does `AutoModel` do differently from loading a specific model class?
2. How does `Trainer` simplify the training loop?
3. When should you use `datasets` instead of loading data manually?
4. What is the benefit of model cards on the Hub?

## 18. Teach-Back Questions

Explain to another person:

- The role of each library in the Hugging Face ecosystem.
- Why `AutoModel`/`AutoTokenizer` are preferred.

## 19. Summary

You explored the Hugging Face ecosystem: transformers, datasets, pipelines, and Trainer, using tiny models to avoid downloads.

## 20. Further Experiment

- Load a real pretrained model from the Hub (requires internet).
- Use `Accelerate` for multi-GPU training.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, transformers, datasets, numpy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
